In [1]:
# Install packages to make sure everything works
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers",
    "datasets",
    "torch",
    "sentence-transformers",
])

import torch
import sentence_transformers

print(f"PyTorch version: {torch.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")

In [2]:
# Import of dataset from huggingface
from datasets import load_dataset
import datasets

# Suppress info/debug logging, unnecessary output
datasets.logging.set_verbosity_error()

mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
# Extract train and test splits
train_data = mnlp_dataset["train"]
test_data = mnlp_dataset["test"]
blind_data = mnlp_dataset["blind"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating blind split:   0%|          | 0/1322 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

In [3]:
# Visualize dataset structure
print("Dataset structure:")
print(mnlp_dataset)

print("\nSample data:")
print(f"{mnlp_dataset['train'][0]}")

Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 2000
    })
    blind: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 1322
    })
    train: Dataset({
        features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer'],
        num_rows: 8000
    })
})

Sample data:
{'wikipedia_title': 'National Air and Space Museum', 'wikidata_id': 'Q752669', 'query': 'when did the smithsonian air and space museum open', 'query_id': 'I9oLuP4O3GOI', 'candidate_chunks': ["The National Air and Space Museum of the Smithsonian Institution, also called the NASM, is a museum in Washington, D.C.. It was established in 1946 as th

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2

In [4]:
from sentence_transformers import SentenceTransformer, models

# Load DistilBERT specifically using the HuggingFace model name
# Create Transformer module from the specific DistilBERT model
word_embedding_model = models.Transformer("distilbert-base-uncased")
# Add pooling layer (mean pooling)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
# Combine into a SentenceTransformer model
model_bert = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# Load MiniLM using SentenceTransformer's built-in support for pre-trained models
model_mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_11948/20616977.py:1: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Implementation of Hit@k metric as:  
$$Hit@k = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1} [r_i \leq k]$$

In [5]:
def hit_at_k(retrieved_indices, ground_truth_indices, k):
    """Hit@k: 1 if any ground truth in top-k results, else 0"""
    top_k_results = retrieved_indices[:k]
    gt_set = set(ground_truth_indices)
    has_hit = any(item in gt_set for item in top_k_results)
    return 1 if has_hit else 0

## [A2]
Report additional performance metrics beyond Hit@k

Implementation of MRR@k metric as:  
$$MRR@k = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i} \cdot \mathbb{1}[rank_i \leq k]$$

In [6]:
def mrr_at_k(retrieved_indices, ground_truth_indices, k):
    """MRR@k: Reciprocal rank of first k relevant items"""

    relevant_items = set(ground_truth_indices)
    
    # Consider only first k elements
    top_k_predictions = retrieved_indices[:k]
    
    for rank, item_id in enumerate(top_k_predictions, start=1):
        if item_id in relevant_items:
            # Found a relevant item at this rank, return reciprocal
            return 1.0 / rank
            
    # No relevant item found in top-k
    return 0.0

Evaluation functions

In [7]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Cosine similarity search utility function
def cosine_similarity_search(query_embedding, chunk_embeddings, top_k=3):
    """Find top-k chunks most similar to query using cosine similarity"""
    similarities = cosine_similarity(query_embedding.reshape(1, -1), chunk_embeddings).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

In [8]:
# Function to evaluate retrieval using both Hit@k and MRR@k metrics
def evaluate_retrieval(queries_embeddings, chunk_embeddings, ground_truths, k_values=[1, 3, 5]):
    results = {}
    for k in k_values:
        hits = [hit_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        mrrs = [mrr_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        results[f'Hit@{k}'] = np.mean(hits)
        results[f'MRR@{k}'] = np.mean(mrrs)
    return results

In [9]:
# Helper function for evaluating on dataset
def evaluate_model_on_dataset(embeddings_dict, model_name="Model"):
    """Evaluate model using pre-computed embeddings and both Hit@k and MRR@k metrics"""
    print(f"Evaluating {model_name} on {len(embeddings_dict)} pre-computed samples...")
    
    hits_per_k = {k: [] for k in [1, 3, 5]}
    mrrs_per_k = {k: [] for k in [1, 3, 5]}
    
    for idx in range(len(embeddings_dict)):
        query_emb = embeddings_dict[idx]['query']
        candidates_emb = embeddings_dict[idx]['candidates']
        answer_pos = embeddings_dict[idx]['answer_pos']
        
        # Compute similarities and ranking
        similarities = cosine_similarity(query_emb.reshape(1, -1), candidates_emb).flatten()
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Use hit_at_k and mrr_at_k functions for evaluation
        for k in [1, 3, 5]:
            hits_per_k[k].append(hit_at_k(ranked_indices, [answer_pos], k))
            mrrs_per_k[k].append(mrr_at_k(ranked_indices, [answer_pos], k))
        
        if (idx + 1) % max(1, len(embeddings_dict) // 5) == 0:
            print(f"  Progress: {(idx + 1) / len(embeddings_dict) * 100:.1f}%")
    
    # Calculate average metrics
    results = {}
    for k in [1, 3, 5]:
        results[f'Hit@{k}'] = np.mean(hits_per_k[k])
        results[f'MRR@{k}'] = np.mean(mrrs_per_k[k])
    
    print(f"Evaluation completed")
    return results

In [10]:
# Pre-compute embeddings for efficient evaluation
def precompute_dataset_embeddings(model, dataset, normalize=True):
    """Pre-compute and cache all embeddings for a dataset"""
    print(f"Pre-computing embeddings for {len(dataset)} examples...")
    embeddings_dict = {}
    
    for idx, example in enumerate(dataset):
        query = example['query']
        candidates = example['candidate_chunks']
        
        # Encode query and candidates with normalization
        query_emb = model.encode(query, convert_to_numpy=True, normalize_embeddings=normalize)
        candidates_emb = model.encode(candidates, convert_to_numpy=True, normalize_embeddings=normalize)
        
        embeddings_dict[idx] = {
            'query': query_emb,
            'candidates': candidates_emb,
            'answer_pos': example['answer_pos']
        }
        
        if (idx + 1) % max(1, len(dataset) // 5) == 0:
            print(f"  Progress: {(idx + 1) / len(dataset) * 100:.1f}%")
    
    print(f"Pre-computed {len(embeddings_dict)} embedding sets\n")
    return embeddings_dict

embeddings_dict = precompute_dataset_embeddings(model_bert, test_data)

Pre-computing embeddings for 2000 examples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Pre-computed 2000 embedding sets



In [11]:
# Evaluate Baseline 1: DistilBERT
print("="*70)
print("BASELINE 1: DistilBERT")
print("="*70 + "\n")

bert_results = evaluate_model_on_dataset(embeddings_dict, "DistilBERT")
print("DistilBERT Results:")
for metric in sorted(bert_results.keys()):
    print(f"  {metric}: {bert_results[metric]:.4f}")

BASELINE 1: DistilBERT

Evaluating DistilBERT on 2000 pre-computed samples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Evaluation completed
DistilBERT Results:
  Hit@1: 0.1765
  Hit@3: 0.4435
  Hit@5: 0.6045
  MRR@1: 0.1765
  MRR@3: 0.2909
  MRR@5: 0.3277


In [16]:
# Evaluate Baseline 2: all-MiniLM-L6-v2
print("="*70)
print("BASELINE 2: all-MiniLM-L6-v2")
print("="*70 + "\n")
mini_embeddings = precompute_dataset_embeddings(model_mini, test_data)
mini_results = evaluate_model_on_dataset(mini_embeddings, "all-MiniLM-L6-v2")
print("all-MiniLM-L6-v2 Results:")
for metric in sorted(mini_results.keys()):
    print(f"  {metric}: {mini_results[metric]:.4f}")

BASELINE 2: all-MiniLM-L6-v2

Pre-computing embeddings for 2000 examples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Pre-computed 2000 embedding sets

Evaluating all-MiniLM-L6-v2 on 2000 pre-computed samples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Evaluation completed
all-MiniLM-L6-v2 Results:
  Hit@1: 0.4480
  Hit@3: 0.7170
  Hit@5: 0.8080
  MRR@1: 0.4480
  MRR@3: 0.5685
  MRR@5: 0.5891


## Task [B2 - Triplets]
Fine-tune DistilBERT using supervised triplet loss to improve semantic search performance. 

Fine-tuning strategy:
1) Create triplet dataset: (query, positive_chunk, negative_chunk)
2) Train using triplet loss to minimize distance between query and correct answer, and maximize distance between query and random other chunks

In [12]:
# Create triplet dataset for fine-tuning
import random
from torch.utils.data import DataLoader
from sentence_transformers import InputExample

print("="*70)
print("PREPARING TRIPLET DATASET")
print("="*70 + "\n")

class TripletDataset:
    """Dataset that generates triplets: (anchor, positive, negative)"""
    def __init__(self, dataset):
        self.triplets = []
        for example in dataset:
            query = example['query']
            candidates = example['candidate_chunks']
            answer_pos = example['answer_pos']
            positive_chunk = candidates[answer_pos]
           
            negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
            negative_pos = random.choice(negative_positions) # Randomly select one negative chunk
            negative_chunk = candidates[negative_pos]
            
            # Create InputExample for sentence-transformers triplet loss
            triplet = InputExample(
                texts=[query, positive_chunk, negative_chunk],
                label=0  # Not used for triplet loss, but required
            )
            self.triplets.append(triplet)
        
        print(f"Created {len(self.triplets)} triplets from {len(dataset)} samples")
    
    def get_triplets(self):
        return self.triplets

# Prepare triplets data from train dataset
triplet_dataset = TripletDataset(train_data)
triplets = triplet_dataset.get_triplets()
train_dataloader = DataLoader(triplets, shuffle=True, batch_size=16)
print(f"DataLoader created: {len(train_dataloader)} batches\n")

PREPARING TRIPLET DATASET

Created 8000 triplets from 8000 samples
DataLoader created: 500 batches



In [14]:
# Train the model with Triplet Loss
from sentence_transformers import losses

print("="*70)
print("FINE-TUNING WITH TRIPLET LOSS")
print("="*70 + "\n")

# Create a copy of the model for fine-tuning
from copy import deepcopy
finetuned_model = deepcopy(model_bert)

# Create new DataLoader with InputExample objects
train_loss = losses.TripletLoss(model=finetuned_model, triplet_margin=0.5)

print(f"Starting fine-tuning...")
print(f"  - Loss: TripletLoss (margin=0.5)")
print(f"  - Epochs: 3")
print(f"  - Batch size: 16")
print(f"  - Total steps: {len(train_dataloader) * 3}\n")

# Fine-tune the model
warmup_steps = int(len(train_dataloader) * 3 * 0.1)

print(f"Training configuration:")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total batches per epoch: {len(train_dataloader)}")
print(f"  Total training steps: {len(train_dataloader) * 3}\n")

finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    optimizer_params={'lr': 2e-5},
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print(f"\nFine-tuning completed")

FINE-TUNING WITH TRIPLET LOSS

Starting fine-tuning...
  - Loss: TripletLoss (margin=0.05)
  - Epochs: 3
  - Batch size: 16
  - Total steps: 1500

Training configuration:
  Warmup steps: 150
  Total batches per epoch: 500
  Total training steps: 1500



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.030994
1000,0.005385
1500,0.001635



Fine-tuning completed


Evaluate Triplet Loss fine-tuned DistilBERT 

In [17]:
# Evaluate fine-tuned model on the same test data
print("="*70)
print("EVALUATING FINE-TUNED MODEL")
print("="*70 + "\n")

finetuned_embeddings = precompute_dataset_embeddings(finetuned_model, test_data)
finetuned_evaluation = evaluate_model_on_dataset(finetuned_embeddings, "Fine-tuned DistilBERT")
print("Fine-tuned Model Results (Hit and MRR):")
for metric in sorted(finetuned_evaluation.keys()):
    print(f"  {metric}: {finetuned_evaluation[metric]:.4f}")

EVALUATING FINE-TUNED MODEL

Pre-computing embeddings for 2000 examples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Pre-computed 2000 embedding sets

Evaluating Fine-tuned DistilBERT on 2000 pre-computed samples...
  Progress: 20.0%
  Progress: 40.0%
  Progress: 60.0%
  Progress: 80.0%
  Progress: 100.0%
Evaluation completed
Fine-tuned Model Results (Hit and MRR):
  Hit@1: 0.5495
  Hit@3: 0.8100
  Hit@5: 0.8815
  MRR@1: 0.5495
  MRR@3: 0.6654
  MRR@5: 0.6816


## Task [B2 - InfoNCE Dynamic Hard Negatives]
Fine-tune DistilBERT with InfoNCE, dynamic hard negatives, in-batch negatives, embedding normalization, and temperature scaling.

Training strategy:
1) At the beginning of each epoch, use the current model to score each query against its candidate chunks.
2) Select the highest-scoring wrong chunk as the hard negative.
3) Train with `MultipleNegativesRankingLoss`, using `(query, positive_chunk, hard_negative_chunk)` examples.
4) Use temperature scaling through the loss scale parameter (`scale = 1 / temperature`).
5) Re-mine hard negatives after every epoch, so the negatives adapt as the model changes.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ============== INFONCE Configuration ==============
INFONCE_BATCH_SIZE = 16
INFONCE_EPOCHS = 5
INFONCE_TEMPERATURE = 0.07
INFONCE_LOGIT_MARGIN = 0.05
INFONCE_HINGE_MARGIN = 0.20
INFONCE_HINGE_WEIGHT = 0.50
INFONCE_HARD_NEGATIVES = 5
INFONCE_TRAIN_SAMPLES = min(4000, len(train_data))

# Hard Negative Dataset Class (uses pre-computed embeddings)
class HardNegativeDataset:
    """Dataset that mines hard negatives using pre-computed embeddings"""
    
    def __init__(self, dataset, embeddings_cache, num_negatives=5, num_samples=None):
        self.dataset = dataset
        self.embeddings_cache = embeddings_cache
        self.num_negatives = num_negatives
        self.num_samples = num_samples
        self.examples = self._build_examples()
    
    def _build_examples(self):
        """Build training examples with hard negatives using pre-computed embeddings"""
        examples = []
        print(f"Mining hard negatives using pre-computed embeddings...")
        
        for i in range(self.num_samples):
            example = self.dataset[i]
            query, candidates, answer_pos = example['query'], example['candidate_chunks'], example['answer_pos']
            positive = candidates[answer_pos]
            
            # Get pre-computed embeddings
            query_emb = self.embeddings_cache[i]['query']
            candidates_embs = self.embeddings_cache[i]['candidates']
            
            # Rank by similarity
            sims = cosine_similarity([query_emb], candidates_embs)[0]
            sorted_idx = np.argsort(sims)[::-1]
            
            # Collect hard negatives (exclude positive)
            hard_negs = [candidates[idx] for idx in sorted_idx if idx != answer_pos][:self.num_negatives]
            
            if not hard_negs:
                continue
            
            # Pad if needed
            while len(hard_negs) < self.num_negatives:
                hard_negs.append(hard_negs[-1])
            
            examples.append(InputExample(texts=[query, positive] + hard_negs[:self.num_negatives]))
            
            if (i + 1) % max(1, self.num_samples // 5) == 0:
                print(f"  Progress: {(i + 1) / self.num_samples * 100:.1f}%")
        
        print(f"Created {len(examples)} training examples with hard negatives\n")
        return examples
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

PRE-COMPUTING EMBEDDINGS



In [19]:
# Fine-tuning with dynamic hard negatives and in-batch negatives
from copy import deepcopy
from sentence_transformers import losses
from torch.utils.data import DataLoader
import numpy as np

print("="*70)
print("B2.0: DYNAMIC HARD NEGATIVES + IN-BATCH NEGATIVES")
print("="*70 + "\n")

dynamic_hn_model = deepcopy(model_bert)
dynamic_hn_epochs = 3
dynamic_hn_batch_size = 16
dynamic_hn_temperature = 0.05
dynamic_hn_scale = 1 / dynamic_hn_temperature

# MultipleNegativesRankingLoss uses the other positives in the same batch as negatives.
# The third text in each InputExample adds one explicit hard negative per query.
dynamic_hn_loss = losses.MultipleNegativesRankingLoss(
    model=dynamic_hn_model,
    scale=dynamic_hn_scale
)

print(f"Temperature: {dynamic_hn_temperature}")
print(f"Loss scale:  {dynamic_hn_scale:.1f}")
print(f"Batch size:   {dynamic_hn_batch_size}\n")

for epoch in range(dynamic_hn_epochs):
    print(f"Epoch {epoch + 1}/{dynamic_hn_epochs}: mining hard negatives with current model...")
    
    # Pre-compute embeddings for train data using current model
    train_embeddings = {}
    print(f"  Pre-computing embeddings...")
    for idx, example in enumerate(train_data):
        query = example['query']
        candidates = example['candidate_chunks']
        query_emb = dynamic_hn_model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
        candidates_emb = dynamic_hn_model.encode(candidates, convert_to_numpy=True, normalize_embeddings=True)
        train_embeddings[idx] = {
            'query': query_emb,
            'candidates': candidates_emb,
            'answer_pos': example['answer_pos']
        }
    
    # Use HardNegativeDataset pattern to mine hard negatives
    hard_neg_dataset = HardNegativeDataset(
        dataset=train_data,
        embeddings_cache=train_embeddings,
        num_negatives=1,  # Single hard negative per query
        num_samples=len(train_data)
    )
    dynamic_hn_examples = hard_neg_dataset.examples
    
    # Compute mining statistics
    mining_stats = []
    for idx, example in enumerate(train_data):
        answer_pos = example['answer_pos']
        candidates_emb = train_embeddings[idx]['candidates']
        query_emb = train_embeddings[idx]['query']
        similarities = cosine_similarity(query_emb.reshape(1, -1), candidates_emb).flatten()
        
        hard_negative_pos = max([i for i in range(len(candidates_emb)) if i != answer_pos], 
                                key=lambda pos: similarities[pos])
        mining_stats.append({
            'positive_score': float(similarities[answer_pos]),
            'hard_negative_score': float(similarities[hard_negative_pos]),
            'hard_negative_pos': int(hard_negative_pos)
        })
    
    avg_positive_score = np.mean([item['positive_score'] for item in mining_stats])
    avg_hard_negative_score = np.mean([item['hard_negative_score'] for item in mining_stats])
    print(f"  Created {len(dynamic_hn_examples)} training examples")
    print(f"  Avg positive score:      {avg_positive_score:.4f}")
    print(f"  Avg hard negative score: {avg_hard_negative_score:.4f}")
    
    dynamic_hn_dataloader = DataLoader(
        dynamic_hn_examples,
        shuffle=True,
        batch_size=dynamic_hn_batch_size
    )
    warmup_steps = max(1, int(len(dynamic_hn_dataloader) * 0.1))
    
    dynamic_hn_model.fit(
        train_objectives=[(dynamic_hn_dataloader, dynamic_hn_loss)],
        epochs=1,
        warmup_steps=warmup_steps,
        show_progress_bar=True
    )

B2.0: DYNAMIC HARD NEGATIVES + IN-BATCH NEGATIVES

Temperature: 0.05
Loss scale:  20.0
Batch size:   16

Epoch 1/3: mining hard negatives with current model...
  Pre-computing embeddings...


KeyboardInterrupt: 

In [ ]:
# Evaluate fine-tuned model on the same test data
print("="*70)
print("EVALUATING FINE-TUNED MODEL")
print("="*70 + "\n")

# Evaluate using pre-computed embeddings
infonce_embeddings = precompute_dataset_embeddings(dynamic_hn_model, test_data)
infonce_evaluation = evaluate_model_on_dataset(infonce_embeddings, 'Info-NCE DistilBERT')
print("Info-NCE Model Results (Hit and MRR):")
for metric in sorted(infonce_evaluation.keys()):
    print(f"  {metric}: {infonce_evaluation[metric]:.4f}")

In [ ]:
# Compare Fine-tuned Model vs DistilBERT Baseline vs MiniLM Baseline
print("="*70)
print("RESULTS COMPARISON: Hit@k and MRR@k Metrics")
print("="*70 + "\n")

comparison_data = {
    'Metric': [],
    'DistilBERT (Baseline 1)': [],
    'MiniLM (Baseline 2)': [],
    'Triplets DistilBERT': [],
    'INFONCE DistilBERT': [],
    'Improvement vs DistilBERT (%)': [],
    'Improvement vs MiniLM (%)': [],
    'Improvement INFONCE vs DistilBERT (%)': [],
    'Improvement INFONCE vs MiniLM (%)': []
}

# Compare Hit@k metrics
print("HIT@K METRICS (Recall-based):")
print("-" * 70)
for metric in ['Hit@1', 'Hit@3', 'Hit@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    infonce_val = infonce_evaluation[metric]
    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100
    improvement_vs_3 = ((infonce_val - baseline_val) / baseline_val) * 100
    improvement_vs_4 = ((infonce_val - baseline2_val) / baseline2_val) * 100
    comparison_data['Metric'].append(metric)
    comparison_data['DistilBERT (Baseline 1)'].append(baseline_val)
    comparison_data['MiniLM (Baseline 2)'].append(baseline2_val)
    comparison_data['Triplets DistilBERT'].append(finetuned_val)
    comparison_data['INFONCE DistilBERT'].append(infonce_val)
    comparison_data['Improvement vs DistilBERT (%)'].append(improvement_vs_1)
    comparison_data['Improvement vs MiniLM (%)'].append(improvement_vs_2)
    comparison_data['Improvement INFONCE vs DistilBERT (%)'].append(improvement_vs_3)
    comparison_data['Improvement INFONCE vs MiniLM (%)'].append(improvement_vs_4)


    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Triplets DistilBERT:   {finetuned_val:.4f}")
    print(f"  INFONCE DistilBERT:   {infonce_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%")
    print(f"  Improvement INFONCE vs DistilBERT: {improvement_vs_3:+.2f}%")
    print(f"  Improvement INFONCE vs MiniLM:     {improvement_vs_4:+.2f}%\n")

# Compare MRR@k metrics
print("MRR@K METRICS (Ranking quality):")
print("-" * 70)
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    infonce_val = infonce_evaluation[metric]

    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100 if baseline_val > 0 else 0
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100 if baseline2_val > 0 else 0
    improvement_vs_3 = ((infonce_val - baseline_val) / baseline_val) * 100 if baseline_val > 0 else 0
    improvement_vs_4 = ((infonce_val - baseline2_val) / baseline2_val) * 100 if baseline2_val > 0 else 0
    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Triplets DistilBERT:   {finetuned_val:.4f}")
    print(f"  INFONCE DistilBERT:   {infonce_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%")
    print(f"  Improvement INFONCE vs DistilBERT: {improvement_vs_3:+.2f}%")
    print(f"  Improvement INFONCE vs MiniLM:     {improvement_vs_4:+.2f}%\n")

In [ ]:
# Visualize Comparison Results: Three-Way Comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Data preparation
metrics = ['Hit@1', 'Hit@3', 'Hit@5']
distilbert_scores = [comparison_data['DistilBERT (Baseline 1)'][i] for i in range(3)]
minilm_scores = [comparison_data['MiniLM (Baseline 2)'][i] for i in range(3)]
finetuned_scores = [comparison_data['Triplets DistilBERT'][i] for i in range(3)]
infonce_scores = [comparison_data['INFONCE DistilBERT'][i] for i in range(3)]

# Plot 1: Three-way side-by-side comparison
x = np.arange(len(metrics))
width = 0.25

axes[0, 0].bar(x - width, distilbert_scores, width, label='DistilBERT (Baseline)', color='steelblue', alpha=0.8)
axes[0, 0].bar(x, minilm_scores, width, label='MiniLM (Baseline)', color='skyblue', alpha=0.8)
axes[0, 0].bar(x + width, finetuned_scores, width, label='Triplets DistilBERT', color='coral', alpha=0.8)
axes[0, 0].bar(x + 2 * width, infonce_scores, width, label='INFONCE DistilBERT', color='lightgreen', alpha=0.8)
axes[0, 0].set_ylabel('Hit@k Score', fontsize=12)
axes[0, 0].set_title('Models Comparison', fontweight='bold', fontsize=13)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend(fontsize=10)
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, (d, m, f, infonce) in enumerate(zip(distilbert_scores, minilm_scores, finetuned_scores, infonce_scores)):
    axes[0, 0].text(i - width, d + 0.02, f'{d:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i, m + 0.02, f'{m:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i + width, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i + 2 * width, infonce + 0.02, f'{infonce:.3f}', ha='center', fontsize=8)

# Plot 2: Improvement vs DistilBERT
improvements_vs_db = comparison_data['Improvement vs DistilBERT (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_db]
axes[0, 1].bar(metrics, improvements_vs_db, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Improvement (%)', fontsize=12)
axes[0, 1].set_title('Triplets DistilBERT vs DistilBERT Baseline', fontweight='bold', fontsize=13)
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_db):
    axes[0, 1].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 3: Improvement vs MiniLM
improvements_vs_ml = comparison_data['Improvement vs MiniLM (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_ml]
axes[1, 0].bar(metrics, improvements_vs_ml, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Improvement (%)', fontsize=12)
axes[1, 0].set_title('Triplets DistilBERT vs MiniLM Baseline', fontweight='bold', fontsize=13)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_ml):
    axes[1, 0].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 4: Improvement vs DistilBERT for INFONCE model
improvements_infonce_vs_db = comparison_data['Improvement INFONCE vs DistilBERT (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_infonce_vs_db]
axes[1, 1].bar(metrics, improvements_infonce_vs_db, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Improvement (%)', fontsize=12)
axes[1, 1].set_title('INFONCE DistilBERT vs DistilBERT Baseline', fontweight='bold', fontsize=13)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_infonce_vs_db):
    axes[1, 1].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

#Plot 5: iMPROVEMENT vs MiniLM for INFONCE model
improvements_infonce_vs_ml = comparison_data['Improvement INFONCE vs MiniLM (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_infonce_vs_ml]
axes[1, 1].bar(metrics, improvements_infonce_vs_ml, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5, bottom=improvements_infonce_vs_db)
axes[1, 1].set_title('INFONCE DistilBERT vs MiniLM Baseline', fontweight='bold', fontsize=13)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_infonce_vs_ml):
    total_improvement = improvements_infonce_vs_db[i] + v
    axes[1, 1].text(i, total_improvement + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 6: Summary table
summary_text = "DETAILED COMPARISON SUMMARY\n" + "="*60 + "\n\n"
summary_text += f"Models:      DistilBERT, MiniLM-L6-v2, Triplets DistilBERT, INFONCE DistilBERT\n"
summary_text += f"Metrics:     Hit@k (Recall) + MRR@k (Ranking)\n"
summary_text += f"Fine-tuning 1: DistilBERT + TripletLoss (3 epochs)\n"
summary_text += f"Loss1:          TripletMarginLoss (margin=0.5)\n"
summary_text += f"Fine-tuning 2: INFONCE DistilBERT + INFONCELoss (3 epochs)\n"
summary_text += f"Loss 2:          InfoNCEHardMarginLoss (temperature=0.07, logit_margin=0.05, hinge_margin=0.20, hinge_weight=0.50)\n"
summary_text += f"Test Dataset:  {len(test_data)} samples\n"
summary_text += f"Training Data: {len(train_data)} triplets\n"
summary_text += "="*60 + "\n\n"

# HIT@K SECTION
summary_text += "HIT@K RESULTS:\n"
hit_k_keys = ['Hit@1', 'Hit@3', 'Hit@5']
for i, metric in enumerate(hit_k_keys):
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {comparison_data['DistilBERT (Baseline 1)'][i]:.4f}\n"
    summary_text += f"    MiniLM:      {comparison_data['MiniLM (Baseline 2)'][i]:.4f}\n"
    summary_text += f"    Triplets DistilBERT:  {comparison_data['Triplets DistilBERT'][i]:.4f}\n"
    summary_text += f"    Δ vs DB:     {comparison_data['Improvement vs DistilBERT (%)'][i]:+.2f}%\n"
    summary_text += f"    Δ vs ML:     {comparison_data['Improvement vs MiniLM (%)'][i]:+.2f}%\n\n"
    summary_text += f"    INFONCE DistilBERT:  {comparison_data['INFONCE DistilBERT'][i]:.4f}\n"
    summary_text += f"    Δ INFONCE vs DB:     {comparison_data['Improvement INFONCE vs DistilBERT (%)'][i]:+.2f}%\n\n"

# MRR@K SECTION
summary_text += "MRR@K RESULTS:\n"
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    db_val = bert_results[metric]
    ml_val = mini_results[metric]
    ft_val = finetuned_evaluation[metric]
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {db_val:.4f}\n"
    summary_text += f"    MiniLM:      {ml_val:.4f}\n"
    summary_text += f"    Triplets DistilBERT:  {ft_val:.4f}\n\n"
    summary_text += f"    INFONCE DistilBERT:  {infonce_evaluation[metric]:.4f}\n\n"

axes[1, 1].text(0.02, 0.95, summary_text, transform=axes[1, 1].transAxes, 
                fontsize=8, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, edgecolor='gray'))
axes[1, 1].axis('off')


plt.tight_layout()
plt.show()